In [ ]:
!pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128 --force-reinstall
!pip install torchtitan

  Attempting uninstall: setuptools
    Found existing installation: setuptools 78.1.0
    Uninstalling setuptools-78.1.0:
      Successfully uninstalled setuptools-78.1.0
  Attempting uninstall: nvidia-nvtx-cu12
    Found existing installation: nvidia-nvtx-cu12 12.8.90
    Uninstalling nvidia-nvtx-cu12-12.8.90:
      Successfully uninstalled nvidia-nvtx-cu12-12.8.90
  Attempting uninstall: nvidia-nvshmem-cu12
    Found existing installation: nvidia-nvshmem-cu12 3.4.5
    Uninstalling nvidia-nvshmem-cu12-3.4.5:
      Successfully uninstalled nvidia-nvshmem-cu12-3.4.5
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.28.9
    Uninstalling nvidia-nccl-cu12-2.28.9:
      Successfully uninstalled nvidia-nccl-cu12-2.28.9
  Attempting u

  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.1.0
    Uninstalling fsspec-2026.1.0:
      Successfully uninstalled fsspec-2026.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cpu requires torch==2.9.0, but you have torch 2.11.0.dev20260131+cu128 which is incompatible.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.11.0.dev20260131+cu128 which is incompatible.
torchvision 0.24.0+cpu requires torch==2.9.0, but you have torch 2.11.0.dev20260131+cu128 which is incompatible.


In [ ]:
import torch
import torchtitan.models.moe as ttmoe


class MoEMLP(torch.nn.Module):
    def __init__(self, config, num_experts=8, top_k=2, device="cuda", dtype=torch.float32):
        super().__init__()
        self.dtype = dtype
        self.device = device  # Store device
        
        ttmoe_args = ttmoe.MoEArgs(
            num_experts=num_experts,
            top_k=top_k,
            num_shared_experts=0,
        )
        
        # Create MoE on the correct device
        self.backbone = ttmoe.MoE(
            ttmoe_args, 
            dim=config.n_embd, 
            hidden_dim=config.n_embd // num_experts
        ).to(device).to(dtype)
        
        # Ensure everything is on the right device and dtype
        self.to(device).to(dtype)

    def forward(self, x):
        x = x.to(self.dtype).to(self.device)
        output = self.backbone(x)
        return output
        # return output.to(torch.float32)

In [ ]:
class Config:
    n_embd = 768

class MLP(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, hidden_dim)
        self.relu = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

print("Config and MLP classes defined successfully.")

## Benchmark: MLP vs MoEMLP

Compare throughput (tokens/sec), latency, and parameter count. Run the cell below after the model cells.

In [ ]:
torch.cuda.is_available()

In [ ]:
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
SEQ_LEN = 2048
HIDDEN = 768
FFN_HIDDEN = 1024
NUM_TOKENS = BATCH_SIZE * SEQ_LEN
ITERATIONS = 100
WARMUP = 10

mlp = torch.compile(MLP(HIDDEN, FFN_HIDDEN, HIDDEN).to(device))
config = Config()
moemlp = torch.compile(MoEMLP(config, num_experts=16, top_k=2, device=device).to(device))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def benchmark(name, model, x, iterations=ITERATIONS, warmup=WARMUP, backward=False):
    model.train() if backward else model.eval()
    for _ in range(warmup):
        out = model(x)
        if isinstance(out, tuple):
            out = out[0]
        if backward:
            out.sum().backward()
            model.zero_grad()
            if x.grad is not None:
                x.grad.zero_()
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iterations):
        out = model(x)
        if isinstance(out, tuple):
            out = out[0]
        if backward:
            out.sum().backward()
            model.zero_grad()
            if x.grad is not None:
                x.grad.zero_()
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    avg_ms = (elapsed / iterations) * 1000
    tps = NUM_TOKENS / (elapsed / iterations)
    mode = "Fwd+Bwd" if backward else "Fwd"
    print(f"  {name} ({mode}): {avg_ms:.2f} ms  |  {tps:,.0f} tokens/sec")
    return tps

x_mlp = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN, device=device, dtype=torch.float32)
x_moe = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN, device=device, dtype=torch.float32)
x_mlp_bwd = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN, device=device, dtype=torch.float32, requires_grad=True)
x_moe_bwd = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN, device=device, dtype=torch.float32, requires_grad=True)

print(f"--- Benchmark (device={device}, tokens={NUM_TOKENS:,}, hidden={HIDDEN}) ---\nForward only:")
mlp_fwd = benchmark("MLP    ", mlp, x_mlp, backward=False)
moe_fwd = benchmark("MoEMLP ", moemlp, x_moe, backward=False)

print("\nForward + Backward:")
mlp_bwd = benchmark("MLP    ", mlp, x_mlp_bwd, backward=True)
moe_bwd = benchmark("MoEMLP ", moemlp, x_moe_bwd, backward=True)

print("\n--- Parameters ---")
print(f"  MLP:    {count_parameters(mlp) / 1e6:.2f} M")
print(f"  MoEMLP: {count_parameters(moemlp) / 1e6:.2f} M")

print("\n--- Throughput vs MLP ---")
if mlp_fwd > 0:
    print(f"  MoEMLP Fwd:     {moe_fwd / mlp_fwd:.2f}x")
if mlp_bwd > 0:
    print(f"  MoEMLP Fwd+Bwd: {moe_bwd / mlp_bwd:.2f}x")